# Question 16 - 42. Trapping Rain Water

Given `n` non-negative integers representing an elevation map where the width of each bar is `1`, compute how much water it can trap after raining.

**Example 1:**

    Input: height = [0,1,0,2,1,0,1,3,2,1,2,1]
    Output: 6
    Explanation: The above elevation map is represented by array [0,1,0,2,1,0,1,3,2,1,2,1]. In this case, 6 units of rain water are being trapped.

                   █
           █≈≈≈█ █ █ █
       █≈≈≈█ █≈█ █ █ █ █ █
     ─────────────────────────
       0 1 0 2 1 0 1 3 2 1 2 1      (≈ = trapped water)

**Example 2:**

    Input: height = [4,2,0,3,2,5]
    Output: 9

**Constraints:**

- `n == height.length`
- `1 <= n <= 2 * 10^4`
- `0 <= height[i] <= 10^5`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Prefix max / suffix max → two pointers

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force | O(n²) | O(1) |
| 1½ Prefix/suffix max arrays | O(n) | O(n) |
| 2️⃣ Optimized (two pointers) | O(n) | O(1) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"Each bar is width 1. I'll think about the water standing **on top of each bar** separately and add them up."*

- Heights non-negative integers? → **Yes.**
- Water spills off both ends? → **Yes.** The first and last bars can never hold water.
- n < 3 → always 0.

**The one formula that drives everything:**

> water[i] = **min(tallest bar to the left of i, tallest bar to the right of i) − height[i]**, floored at 0.

*Why:* water above bar `i` rises until it spills over the **lower** of the two walls that enclose it. The left wall is the max on the left and the right wall is the max on the right.

### Step 1 · Brute force

**Idea:** Apply the formula directly. For every `i`, scan left for the max and scan right for the max.

🗣️ *"Water at each index is the smaller of the highest wall on its left and on its right, minus its own height. Brute force computes both maxes by scanning for every index, which is O(n²). The repeated work is obvious: neighbouring indices recompute nearly the same maxes."*

In [1]:
class SolutionBrute:
    def trap(self, height: list[int]) -> int:
        n = len(height)
        total = 0
        for i in range(n):
            left_max = max(height[: i + 1])     # includes i itself
            right_max = max(height[i:])
            total += min(left_max, right_max) - height[i]
        return total

#### Step 1½ · Precompute the maxes (prefix / suffix arrays)

`left_max[i] = max(left_max[i−1], height[i])` is a running max from the left, and `right_max` is the same from the right. Each one is a single pass.

🗣️ *"I'll precompute both running maxima in two passes, then do one pass to sum. That's O(n) time but O(n) extra space. Can I avoid storing the arrays?"*

In [2]:
class SolutionPrefix:
    def trap(self, height: list[int]) -> int:
        n = len(height)
        left_max = [0] * n
        right_max = [0] * n
        left_max[0] = height[0]
        for i in range(1, n):
            left_max[i] = max(left_max[i - 1], height[i])
        right_max[-1] = height[-1]
        for i in range(n - 2, -1, -1):
            right_max[i] = max(right_max[i + 1], height[i])
        return sum(min(left_max[i], right_max[i]) - height[i] for i in range(n))

### Step 2 · Optimize

**Bottleneck:** We store two full arrays, but at each index we only use the **smaller** of the two values.

**Key insight:** Put a pointer at each end, `L` and `R`, and track `left_max` (max of `height[0..L]`) and `right_max` (max of `height[R..n−1]`).

If `left_max < right_max`:
- For index `L`, the true right-side max is **at least** `right_max` (there may be even taller bars in the unseen middle, but never shorter).
- So `min(true_left, true_right) = left_max`. **The left side is the bottleneck and we already know it exactly.**
- → Water at `L` = `left_max − height[L]`. Settle it and move `L` right.

Otherwise the symmetric argument settles `R`.

**Always move the side with the smaller max**, because that side's answer is already determined.

🗣️ *"Water at an index depends only on the smaller of its two walls. With two pointers, if my left max is smaller than my right max, then the left index's water is decided by the left max: whatever lies between can only make the right wall taller, not shorter. So I add the water there and move left. Otherwise I do the same from the right. One pass, O(1) space."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| Scan both sides for each index | ❌ | O(n²). It recomputes maxima. |
| Prefix/suffix max arrays | ✅ (clear) | O(n), and the easiest to explain and prove. Costs two O(n) arrays. |
| Monotonic **stack** (fill by horizontal layers) | ✅ (alternative) | O(n) time, O(n) stack. Computes water layer by layer when a taller bar closes a basin. Good to know, but harder to get right live. |
| **Two pointers + running maxes** | ✅ (best) | O(n) time, **O(1)** space. Relies on "the smaller wall decides". |
| Segment tree / sparse table for range max | ❌ | Overkill. We only need prefix and suffix maxima, not arbitrary ranges. |

**Interview strategy:** say the formula first, then give the prefix-array version (which proves you understand it), then upgrade to two pointers.

In [3]:
class Solution:
    def trap(self, height: list[int]) -> int:
        left, right = 0, len(height) - 1
        left_max = right_max = 0
        total = 0
        while left < right:
            left_max = max(left_max, height[left])
            right_max = max(right_max, height[right])
            if left_max < right_max:
                # the left wall is the bottleneck for index `left`
                total += left_max - height[left]
                left += 1
            else:
                # the right wall is the bottleneck for index `right`
                total += right_max - height[right]
                right -= 1
        return total

### Step 3 · Toy example walkthrough

`height = [4, 2, 0, 3, 2, 5]` (expected 9)

| step | L | R | h[L] | h[R] | left_max | right_max | which side? | water added | total |
|---|---|---|---|---|---|---|---|---|---|
| 1 | 0 | 5 | 4 | 5 | 4 | 5 | 4 < 5 → L | 4−4 = 0 | 0 |
| 2 | 1 | 5 | 2 | 5 | 4 | 5 | L | 4−2 = **2** | 2 |
| 3 | 2 | 5 | 0 | 5 | 4 | 5 | L | 4−0 = **4** | 6 |
| 4 | 3 | 5 | 3 | 5 | 4 | 5 | L | 4−3 = **1** | 7 |
| 5 | 4 | 5 | 2 | 5 | 4 | 5 | L | 4−2 = **2** | 9 |
| stop | 5 | 5 | | | | | L == R | | **9** ✅ |

The left wall (height 4) is always the lower one, so every middle bar fills up to 4. The tall right wall (5) only guarantees that the water doesn't spill out on the right.

Check with the formula: index 1 → min(4,5) − 2 = 2; index 2 → 4; index 3 → 1; index 4 → 2. The sum is 9 ✅

In [4]:
cases = [
    ([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1], 6),
    ([4, 2, 0, 3, 2, 5], 9),
    ([1], 0),
    ([2, 0], 0),
    ([3, 0, 3], 3),
    ([1, 2, 3, 4], 0),            # monotone -> nothing trapped
    ([5, 4, 1, 2], 1),
    ([0, 0, 0], 0),
]
for h, expected in cases:
    for Impl in (SolutionBrute, SolutionPrefix, Solution):
        assert Impl().trap(h) == expected, (Impl.__name__, h)
    print(h, "->", expected)

import random
for _ in range(500):
    h = [random.randint(0, 6) for _ in range(random.randint(1, 12))]
    assert Solution().trap(h) == SolutionBrute().trap(h), h
print("All tests passed ✅")

[0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1] -> 6
[4, 2, 0, 3, 2, 5] -> 9
[1] -> 0
[2, 0] -> 0
[3, 0, 3] -> 3
[1, 2, 3, 4] -> 0
[5, 4, 1, 2] -> 1
[0, 0, 0] -> 0
All tests passed ✅


### Step 4 · Wrap up

| | Time | Space | Why |
|---|---|---|---|
| Brute force | O(n²) | O(1) | Re-scan both sides for each index |
| Prefix/suffix arrays | O(n) | O(n) | Three linear passes, two arrays |
| Monotonic stack | O(n) | O(n) | Each bar pushed/popped once |
| **Two pointers** | **O(n)** | **O(1)** | Each step settles one index for good |

**Complexity, explained:** each loop iteration moves either `left` or `right` one step inward. They start n−1 apart and stop when they meet, so there are exactly n−1 iterations with O(1) work each, which is O(n). We keep five integers, so space is O(1).

**Edge cases:** n ≤ 2 → 0. Monotone heights → 0. Flat zeros → 0. Equal tall walls (`[3,0,3]`) → 3.

**Follow-ups:**
- *2D version* (LC 407 Trapping Rain Water II) → min-heap from the border inward (Dijkstra-like). The "lowest wall decides" idea carries over.
- *Container With Most Water* (Q27) looks similar but is different: there you pick **two** lines and the bars in between don't matter.

🗣️ **30-second recap:** *"Water above each bar is min(max-left, max-right) minus its height. Brute force computes those maxes per index in O(n²). Precomputing prefix and suffix maxima gives O(n) time and O(n) space. With two pointers I only need the smaller side's max: if left max is smaller, the left index's water is already determined, so I add it and move left; otherwise I do the same on the right. O(n) time, O(1) space."*